# Group Assignment - Part A: Sentence Probability using Bigram Language Models (Q4)

This notebook implements Unsmoothed Bigram (MLE) and Smoothed Bigram (Laplace) language models using the provided Data_3.txt. The implementation follows the workflow introduced in Lab 03 (Tokenization) and Lab 07 (Language Modelling).

## Environment Setup

### Environment Dependencies

First, import the required libraries for tokenization, bigram generation, language model construction and probability computation.

In [11]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.util import bigrams, everygrams
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.lm import MLE, Laplace
import pandas as pd

nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/jeffrey1108/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Step 4.1: Load the Training Corpus

The training corpus is loaded from Data_3.txt. Each sentence already contains <s> and </s> sentence boundary markers.

In [12]:

with open("Data_3.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

print(raw_text)

corpus = []

for line in raw_text.splitlines():
    if line.startswith("<s>"):
        corpus.append(line)

print("\nTraining Corpus")
for sentence in corpus:
    print(sentence)

Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a book by Danielle </s>

Training Corpus
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>
<s> I read a book by Danielle </s>


## Step 4.2: Text Pre-processing

Each sentence is tokenized while preserving the sentence boundary markers.

In [13]:
tokenized_text = [sentence.split() for sentence in corpus]

print("Tokenized Sentences\n")

for i, sentence in enumerate(tokenized_text, start=1):
    print(f"Sentence {i}:")
    print(sentence)
    print()

Tokenized Sentences

Sentence 1:
['<s>', 'He', 'read', 'a', 'book', '</s>']

Sentence 2:
['<s>', 'I', 'read', 'a', 'different', 'book', '</s>']

Sentence 3:
['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']

Sentence 4:
['<s>', 'I', 'read', 'a', 'book', 'by', 'Danielle', '</s>']



# Step 4.3: Generate Bigrams

Each tokenized sentence is converted into a sequence of consecutive word pairs (bigrams). These bigrams form the basis of the statistical language model used in this assignment.

In [14]:
print("Generated Bigrams\n")

for i, sentence in enumerate(tokenized_text, start=1):

    print(f"Sentence {i}")

    bg = list(bigrams(sentence))

    print(bg)
    print()

Generated Bigrams

Sentence 1
[('<s>', 'He'), ('He', 'read'), ('read', 'a'), ('a', 'book'), ('book', '</s>')]

Sentence 2
[('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'different'), ('different', 'book'), ('book', '</s>')]

Sentence 3
[('<s>', 'He'), ('He', 'read'), ('read', 'a'), ('a', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]

Sentence 4
[('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]



# Step 4.4: Generate Everygrams

everygrams are generated before constructing the language model. This allows the language model to learn the required n-gram information during training.

In [15]:
print("Everygrams\n")

for sentence in tokenized_text:

    padded = list(pad_both_ends(sentence, n=2))

    print(list(everygrams(padded, max_len=2)))
    print()

Everygrams

[('<s>',), ('<s>', '<s>'), ('<s>',), ('<s>', 'He'), ('He',), ('He', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'book'), ('book',), ('book', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]

[('<s>',), ('<s>', '<s>'), ('<s>',), ('<s>', 'I'), ('I',), ('I', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'different'), ('different',), ('different', 'book'), ('book',), ('book', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]

[('<s>',), ('<s>', '<s>'), ('<s>',), ('<s>', 'He'), ('He',), ('He', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'book'), ('book',), ('book', 'by'), ('by',), ('by', 'Danielle'), ('Danielle',), ('Danielle', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]

[('<s>',), ('<s>', '<s>'), ('<s>',), ('<s>', 'I'), ('I',), ('I', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'book'), ('book',), ('book', 'by'), ('by',), ('by', 'Danielle'), ('Danielle',), ('Danielle', '</s>'), ('</s>',), ('</s>', '</s>'), ('</s>',)]



# Step 4.5: Train the Unsmoothed Bigram Language Model (MLE)

The **Maximum Likelihood Estimation (MLE)** language model is trained using the tokenized training corpus. This model estimates conditional probabilities directly from the observed bigram frequencies.

In [16]:
n = 2

train_data, vocab_data = padded_everygram_pipeline(n, tokenized_text)

mle_model = MLE(n)

mle_model.fit(train_data, vocab_data)

print("Unsmoothed Bigram Language Model trained successfully.")

Unsmoothed Bigram Language Model trained successfully.


# Step 4.6: Train the Smoothed Bigram Language Model (Laplace)

The **Laplace (Add-One) Language Model** is trained using the same training corpus. Laplace smoothing assigns a non-zero probability to unseen bigrams, thereby reducing the zero-frequency problem.

In [17]:
train_data, vocab_data = padded_everygram_pipeline(n, tokenized_text)

laplace_model = Laplace(n)

laplace_model.fit(train_data, vocab_data)

print("Laplace Bigram Language Model trained successfully.")

Laplace Bigram Language Model trained successfully.


# Step 4.7: Prepare the Target Sentence

The target sentence is tokenized, padded with sentence boundary markers, and converted into bigrams before calculating its sentence probability.

In [18]:
target_sentence = "I read a book by Danielle"

target_tokens = word_tokenize(target_sentence)

print("Target Tokens")

print(target_tokens)

padded_target = list(pad_both_ends(target_tokens, n=2))

print("\nPadded Sentence")

print(padded_target)

target_bigrams = list(bigrams(padded_target))

print("\nTarget Sentence Bigrams")

print(target_bigrams)

Target Tokens
['I', 'read', 'a', 'book', 'by', 'Danielle']

Padded Sentence
['<s>', 'I', 'read', 'a', 'book', 'by', 'Danielle', '</s>']

Target Sentence Bigrams
[('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]


# Step 4.8: Compute Sentence Probability

The probability of the target sentence is computed by multiplying the conditional probability of every bigram using both the Unsmoothed Bigram (MLE) and Smoothed Bigram (Laplace) language models.

In [20]:
mle_probability = 1.0
laplace_probability = 1.0

results = []

for bg in target_bigrams:

    history = (bg[0],)

    word = bg[1]

    mle_score = mle_model.score(word, history)

    laplace_score = laplace_model.score(word, history)

    mle_probability *= mle_score

    laplace_probability *= laplace_score

    results.append([
        history[0],
        word,
        mle_score,
        laplace_score
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "History",
        "Word",
        "MLE Probability",
        "Laplace Probability"
    ]
)

results_df

,History,Word,MLE Probability,Laplace Probability
0,<s>,I,0.25,0.157895
1,I,read,1.00,0.230769
2,read,a,1.00,0.333333
3,a,book,0.75,0.266667
4,book,by,0.50,0.200000
5,by,Danielle,1.00,0.230769
6,Danielle,</s>,1.00,0.230769


# Step 4.9: Display Sentence Probabilities

The overall sentence probability is obtained by multiplying all conditional probabilities generated by each language model.

In [22]:
print("Vocabulary Size:", len(laplace_model.vocab))

print("\nSentence:")

print("<s> I read a book by Danielle </s>")

print("\nUnsmoothed (MLE) Sentence Probability")

print(mle_probability)

print("\nSmoothed (Laplace) Sentence Probability")

print(laplace_probability)

Vocabulary Size: 11

Sentence:
<s> I read a book by Danielle </s>

Unsmoothed (MLE) Sentence Probability
0.09375

Smoothed (Laplace) Sentence Probability
3.449680185899432e-05


# Step 4.10: Compare the Results

The table below summarises the differences between the two language models.

# Conclusion

This notebook successfully demonstrates the implementation of both **Unsmoothed Bigram (Maximum Likelihood Estimation)** and **Smoothed Bigram (Laplace)** language models using the training corpus provided in **Data_3.txt**. The sentence probability was computed by multiplying the conditional probability of each bigram in the target sentence. The results show that the Laplace model produces a lower probability because smoothing distributes probability mass across the vocabulary while preventing zero probabilities for unseen word pairs.

Overall, the implementation follows the workflow introduced in the laboratory sessions while extending the solution to fully satisfy the requirements of Assignment Q4.